# 1. Introducción a los Agentes LLM

En los últimos años, los Large Language Models (LLMs) como GPT han demostrado capacidades sorprendentes para generar texto, responder preguntas y ayudar en tareas complejas.

Sin embargo, un LLM tradicional tiene limitaciones importantes:

- No puede acceder automáticamente a información actualizada.
- No puede ejecutar código.
- No puede consultar bases de datos.
- No puede interactuar directamente con herramientas externas.
- No puede planificar tareas complejas de múltiples pasos.

Para resolver estas limitaciones, surgieron los **agentes LLM**.

## ¿Qué es un agente?

Un agente es un sistema basado en un LLM que puede:

1. Razonar sobre una tarea.
2. Decidir qué herramientas utilizar.
3. Ejecutar acciones.
4. Observar resultados.
5. Continuar razonando hasta resolver el problema.


## LLM tradicional vs Agente

### LLM tradicional

Usuario → Prompt → LLM → Respuesta

El modelo solamente genera texto.

### Agente

Usuario → Agente LLM
                ↓
        Herramientas externas
                ↓
        Observaciones/resultados
                ↓
        Respuesta final

El modelo puede interactuar con el mundo exterior a través de herramientas.


## Ejemplos de herramientas

Un agente puede utilizar herramientas como:

- Calculadoras
- APIs web
- Motores de búsqueda
- Bases de datos SQL
- Sistemas RAG
- Ejecutores de código Python
- Sistemas de archivos
- APIs empresariales

Un agente no solamente "habla".

Un agente:
- razona,
- decide,
- actúa,
- observa,
- y vuelve a razonar.

## Frameworks modernos

En este tutorial utilizaremos principalmente:

- LangChain
- LangGraph
- OpenAI API


## Plan del tutorial de hoy

- A- LangChain: Construir un primer agente
- B- Introducción a LangGraph: construir un agente con un sistema de decisión basado en grafo
- C- Constuir un flujo de trabajo con agentes (_agentic workflows_) 

---

# 2. Instalación del entorno

In [ ]:
#!pip install -q langchain langgraph langchain-openai

In [ ]:
!python -V

In [ ]:
# Configuración de API Key
# IMPORTANTE: nunca escribas la clave directamente aquí (queda expuesta al subir el notebook).
# Defínela como variable de entorno o en un archivo .env (ignorado por git).
import os

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError(
        "Falta OPENAI_API_KEY. Defínela como variable de entorno "
        "o crea un archivo .env con: OPENAI_API_KEY=tu_clave"
    )

---

# 3. A - Construir un primer agente con LangChain

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

respuesta = llm.invoke("Calcula 25 * 37 y dime el valor actual de la conversion USD a CLP")

print(respuesta.content)

/home/matthieu/Documents/trabajo/docencia/2026/MAD/sesion1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Primero, calculemos el producto de 25 y 37:

\[ 25 \times 37 = 925 \]

En cuanto a la conversión de USD a CLP (pesos chilenos), el tipo de cambio puede variar. A partir de mi última actualización, el tipo de cambio estaba alrededor de 800-900 CLP por 1 USD, pero te recomiendo que verifiques el tipo de cambio actual en una fuente confiable, como un banco o un sitio web financiero.

Si necesitas un cálculo específico, por favor proporciona el tipo de cambio actual y puedo ayudarte a hacer la conversión.


## Limitaciones de un LLM tradicional

Aunque el modelo puede responder preguntas, sigue teniendo limitaciones importantes.

Por ejemplo:
- no puede consultar internet directamente,
- no puede calcular con precisión absoluta,
- no puede ejecutar programas,
- no puede interactuar con APIs.

## Introducción a las herramientas (Tools)

Una herramienta es simplemente una función que el agente puede utilizar.

Por ejemplo:
- una calculadora,
- un buscador web,
- una API,
- una consulta SQL.

Comenzaremos creando una herramienta muy simple.

In [8]:
from langchain.tools import tool

`eval()` es una función de Python que **evalúa una cadena de texto como si fuera una expresión Python**.
por ejemplo:
- eval("2 + 3 * 4")  # 14
- eval("len([1,2,3])")

In [22]:
@tool
def calculadora(expresion: str) -> str:
    """
    Solo evalúa expresiones matemáticas simples.
    """
    
    resultado = eval(expresion)
    
    return str(resultado)

In [10]:
calculadora.invoke("25 * 12 + 3")

'303'

In [11]:
calculadora.invoke("__import__('math').sqrt(16)")

'4.0'

## Del LLM al agente

Hasta ahora tenemos un modelo de lenguaje y una herramienta.

La idea fundamental de un agente es:

> Permitir que el LLM decida cuándo usar herramientas.

Es decir:

1. El usuario hace una pregunta.
2. El modelo razona.
3. El modelo decide usar una herramienta.
4. Observa el resultado.
5. Genera una respuesta final.

Este ciclo es la base de los agentes modernos.

Nuestro primer agente simple:
- tendrá acceso a una calculadora,
- podrá decidir cuándo usarla,
- y responderá automáticamente.

In [12]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool

Definimos las herramientas

In [13]:
tools = [calculadora]

Creamos el agente

In [14]:
agent = create_agent(
    model=llm,
    tools=tools
)

Ejecutamos el agente

In [25]:
result = agent.invoke({
    "messages": [HumanMessage(content="Calcula 25 * 37")],
})

print(result["messages"][-1].content)

El resultado de \( 25 \times 37 \) es 925.


Inspección completa (debug). Esto muestra todo el flujo interno del agente:

In [26]:
print(result)

{'messages': [HumanMessage(content='Calcula 25 * 37', additional_kwargs={}, response_metadata={}, id='d8aa0b98-25ca-4ccc-82ee-586067d800ee'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 53, 'total_tokens': 73, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_997b5b4ee9', 'id': 'chatcmpl-DsSHBbMm7CfcH8OeEtTyhvYi28Vb1', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019edfc6-4f2b-7223-8ddb-ea5981bd011f-0', tool_calls=[{'name': 'calculadora', 'args': {'expresion': '25 * 37'}, 'id': 'call_7HYN7Zf8q3dlcPvTJXNITEAF', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 53, 'output_tokens':

Visualización del comportamiento del agente con una funcion permite ver el proceso completo:
- input del usuario
- decisión de herramienta
- ejecución de herramienta
- respuesta final

In [27]:
def explain_agent(result):
    print("\n══════════════════════")
    print("Comportamiento del agente")
    print("══════════════════════\n")

    for i, m in enumerate(result["messages"]):
        print(f"--- STEP {i} ---")
        print(type(m).__name__)

        # input humano
        if hasattr(m, "content") and not hasattr(m, "tool_calls"):
            print(m.content)

        # decisión del agente
        if hasattr(m, "tool_calls") and m.tool_calls:
            for t in m.tool_calls:
                print("DECIDE USAR TOOL:")
                print(f"   ➜ {t['name']}({t['args']})")

        # resultado tool
        if m.__class__.__name__ == "ToolMessage":
            print("RESULTADO TOOL:", m.content)

        print()

    print("══════════════════════")
    print("RESPUESTA FINAL:")
    print(result["messages"][-1].content)
    print("══════════════════════")

In [28]:
explain_agent(result)


══════════════════════
Comportamiento del agente
══════════════════════

--- STEP 0 ---
HumanMessage
Calcula 25 * 37

--- STEP 1 ---
AIMessage
DECIDE USAR TOOL:
   ➜ calculadora({'expresion': '25 * 37'})

--- STEP 2 ---
ToolMessage
925
RESULTADO TOOL: 925

--- STEP 3 ---
AIMessage

══════════════════════
RESPUESTA FINAL:
El resultado de \( 25 \times 37 \) es 925.
══════════════════════


## El paradigma ReAct

Muchos agentes modernos utilizan una estrategia llamada ReAct:

Reasoning + Acting

Es decir:

1. El modelo razona.
2. Ejecuta acciones.
3. Observa resultados.
4. Continúa razonando.

---

# 4. Actualización del agente con nuevo herramienta

Vamos a crear una tool que consulta el tipo de cambio USD/CLP.

Usaremos una API gratuita: https://open.er-api.com (Exchange Rate API)

In [29]:
import requests

Nuevo tool: Tasa 1 USD = ? CLP en tiempo real

In [30]:
@tool
def usd_a_clp() -> str:
    """
    Obtiene el valor actual del dólar (USD) en pesos chilenos (CLP).
    """
    url = "https://open.er-api.com/v6/latest/USD"
    response = requests.get(url)
    data = response.json()

    rate = data["rates"]["CLP"]

    return f"1 USD = {rate} CLP"

In [31]:
tools = [calculadora, usd_a_clp]

agent = create_agent(
    model=llm,
    tools=tools
)

In [32]:
result = agent.invoke({
    "messages": [HumanMessage(content="¿Cuál es el valor actual del dólar en Chile?")]
})

print(result["messages"][-1].content)

El valor actual del dólar en Chile es de 1 USD = 890.63 CLP.


In [33]:
explain_agent(result)


══════════════════════
Comportamiento del agente
══════════════════════

--- STEP 0 ---
HumanMessage
¿Cuál es el valor actual del dólar en Chile?

--- STEP 1 ---
AIMessage
DECIDE USAR TOOL:
   ➜ usd_a_clp({})

--- STEP 2 ---
ToolMessage
1 USD = 890.63154 CLP
RESULTADO TOOL: 1 USD = 890.63154 CLP

--- STEP 3 ---
AIMessage

══════════════════════
RESPUESTA FINAL:
El valor actual del dólar en Chile es de 1 USD = 890.63 CLP.
══════════════════════


In [34]:
result = agent.invoke({
    "messages": [HumanMessage(content="Si tengo 1.000 CLP, ¿a cuánto USD corresponde?", )]
})

print(result["messages"][-1].content)

Si tienes 1.000 CLP, eso corresponde aproximadamente a 1.12 USD.


In [35]:
explain_agent(result)


══════════════════════
Comportamiento del agente
══════════════════════

--- STEP 0 ---
HumanMessage
Si tengo 1.000 CLP, ¿a cuánto USD corresponde?

--- STEP 1 ---
AIMessage
DECIDE USAR TOOL:
   ➜ usd_a_clp({})

--- STEP 2 ---
ToolMessage
1 USD = 890.63154 CLP
RESULTADO TOOL: 1 USD = 890.63154 CLP

--- STEP 3 ---
AIMessage
DECIDE USAR TOOL:
   ➜ calculadora({'expresion': '1000 / 890.63154'})

--- STEP 4 ---
ToolMessage
1.122798772655188
RESULTADO TOOL: 1.122798772655188

--- STEP 5 ---
AIMessage

══════════════════════
RESPUESTA FINAL:
Si tienes 1.000 CLP, eso corresponde aproximadamente a 1.12 USD.
══════════════════════


## Limitaciones de los agentes clásicos

Aunque este enfoque es muy útil para aprender, los agentes clásicos de LangChain tienen varios problemas:

- poca estabilidad,
- loops infinitos,
- dificultad de debugging,
- poco control del flujo,
- ejecución impredecible.

Por esta razón, los sistemas modernos están migrando hacia:

- workflows explícitos,
- grafos de ejecución,
- LangGraph.

Más adelante construiremos agentes modernos utilizando grafos y estados explícitos.

---

# 4. B - Introducción a LangGraph: Construir un agente con un sistema de decisión basado en grafo

LangGraph es una librería para construir sistemas LLM basados en grafos y estados explícitos.

En lugar de tener un agente completamente libre, definimos:

- nodos,
- estados,
- transiciones,
- y reglas de ejecución.

La idea es que un sistema basado en grafos es: más controlable, más estable, más observable, y más fácil de debuggear.

## Componentes principales de un agente que usa LangGraph

### Estado

Información compartida del sistema.

Por ejemplo:
- historial de mensajes,
- resultados intermedios,
- memoria,
- contexto.

### Nodos

Funciones que realizan tareas.

Por ejemplo:
- llamar al LLM,
- ejecutar herramientas,
- recuperar documentos,
- validar respuestas.

### Aristas (edges)

Definen cómo fluye la ejecución entre nodos.


LangGraph NO elimina el agente. LangGraph estructura, controla y estabiliza el comportamiento del agente.

## Primer grafo simple

En este primer ejemplo construiremos el sistema más simple posible en LangGraph: un grafo con un solo nodo que contiene un modelo de lenguaje (LLM).

La idea clave es entender que en LangGraph **todo es un grafo de estados**, incluso un chatbot básico.

Este sistema:

- Recibe un mensaje del usuario
- Lo envía a un LLM
- Devuelve la respuesta del modelo

En forma de flujo:
input --> LLM --> output

LangGraph utiliza un objeto de estado compartido entre nodos.

Este estado representa toda la información que fluye por el grafo. En este caso, solo contiene:
- message: el mensaje del usuario

In [ ]:
#!pip install -q langgraph

In [36]:
from typing import TypedDict

class State(TypedDict):
    message: str


In [37]:
from langgraph.graph import StateGraph

graph_builder = StateGraph(State)

Un nodo es simplemente una función que recibe el estado, procesa información y devuelve una actualización del estado

In [38]:
def chatbot_node(state: State):
    
    response = llm.invoke(state["message"])
    
    return {
        "message": response.content
    }

In [39]:
graph_builder.add_node(
    "chatbot",
    chatbot_node
)

graph_builder.set_entry_point("chatbot")
graph_builder.set_finish_point("chatbot")

graph = graph_builder.compile()

In [40]:
resultado = graph.invoke({
    "message": "¿Cuál es el valor de 1.000 CLP en USD?"
})

resultado


{'message': 'El valor del peso chileno (CLP) en dólares estadounidenses (USD) puede variar diariamente debido a las fluctuaciones del mercado cambiario. Para obtener la tasa de cambio más actualizada, te recomendaría consultar un sitio web financiero o una plataforma de cambio de divisas. Sin embargo, como referencia, en octubre de 2023, la tasa de cambio estaba aproximadamente en el rango de 800 a 900 CLP por 1 USD. Esto significa que 1.000 CLP equivaldrían a aproximadamente 1.11 a 1.25 USD, dependiendo de la tasa exacta en ese momento. Te sugiero verificar la tasa actual para obtener un valor preciso.'}

Hasta ahora, el grafo solo puede responder usando un LLM. Necesitamos que el sistema pueda:
- Pensar (LLM)
- Actuar (tools)
- Decidir rutas de ejecución (lógica del grafo)

El grafo se transforma en un sistema de decisión.

In [41]:
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

from typing import TypedDict, Annotated

In [42]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

Annotated es una forma de añadir metadatos a un tipo de dato en Python.

Con Annotated[list, add_messages] la lista no se reemplaza, sino que se va acumulando con mensajes nuevo. En otras palabras, State.messages es el **historial de chat** del agente.

In [43]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

El LLM no solo responde texto, también puede llamar funciones externas.

In [44]:
tools = [calculadora, usd_a_clp]
llm_with_tools = llm.bind_tools(tools)

El nodo agente representa el “cerebro” del agente.

- Recibe el estado (historial de mensajes)
- El LLM responde directamente o puede pedir una herramienta

In [45]:
def agent_node(state: State):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

Nodo que ejecuta una herramienta:

In [46]:
tool_node = ToolNode(tools)

Lógica de decisión (routing):
- si el último  mensaje pidió usar una herramienta, se va al nodo "tools"
- sino, se termina el grafo

In [47]:
def should_continue(state: State):
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return END

Construcción del grafo:
- 2 tipos de nodos: agent y tools
- si el agente pide una herramienta, se va al nodo tools, sino se termina
- cuando se termina el nodo "tools", se vuelve al agente "nodo"

In [48]:
graph = StateGraph(State)

graph.add_node("agent", agent_node)
graph.add_node("tools", tool_node)

graph.set_entry_point("agent")

graph.add_conditional_edges(
    "agent",
    should_continue,
    {
        "tools": "tools",
        END: END
    }
)

graph.add_edge("tools", "agent")

app = graph.compile()

In [49]:
result = app.invoke({
    "messages": [
        {"role": "user", "content": "¿Cuál es el valor de 1.000 CLP en USD?"}
    ]
})

result["messages"][-1].content

'El valor de 1.000 CLP es aproximadamente 1.12 USD.'

In [50]:
explain_agent(result)


══════════════════════
Comportamiento del agente
══════════════════════

--- STEP 0 ---
HumanMessage
¿Cuál es el valor de 1.000 CLP en USD?

--- STEP 1 ---
AIMessage
DECIDE USAR TOOL:
   ➜ usd_a_clp({})

--- STEP 2 ---
ToolMessage
1 USD = 890.63154 CLP
RESULTADO TOOL: 1 USD = 890.63154 CLP

--- STEP 3 ---
AIMessage
DECIDE USAR TOOL:
   ➜ calculadora({'expresion': '1000 / 890.63154'})

--- STEP 4 ---
ToolMessage
1.122798772655188
RESULTADO TOOL: 1.122798772655188

--- STEP 5 ---
AIMessage

══════════════════════
RESPUESTA FINAL:
El valor de 1.000 CLP es aproximadamente 1.12 USD.
══════════════════════


In [51]:
try:
    result = app.invoke(
        {
            "messages": [
                {"role": "user", "content": "¿Cuál es el valor de 1.000 CLP en USD?"}
            ]
        },
        config={"recursion_limit": 10} ## EVITA LOOP INFINITO O LARGO
    )

    print(result["messages"][-1].content)

except Exception as e:
    print("No se llegó a una respuesta del agente")
    print("Error técnico:", str(e))

El valor de 1.000 CLP es aproximadamente 1.12 USD.


---

# 5. C - Siguiente etapa: Construir un flujo de trabajo con agentes (_Agentic Workflows_)

En el punto en que estamos, ya construimos un agente con LangGraph que:

- usa un LLM
- puede llamar herramientas (tools)
- decide cuándo usar esas herramientas
- mantiene un estado de conversación

Esto ya es un sistema muy potente, pero también tiene el límite del enfoque “agente simple”.

Aparece una idea más importante: construir **flujos de trabajo con agentes** (agentic workflows).

El modelo "agente simple" tiene varias limitaciones:
- el LLM decide cuándo usar tools
- el routing depende de `tool_calls`
- el flujo emerge dinámicamente.

No tenemos garantía de un proceso estable. Esto puede generar loops inesperados, comportamientos inconsistentes, dificultad para depurar y monitorear costos.

**¿Cuál es el problema?**

- El LLM concentra demasiada responsabilidad. El mismo LLM intepreta la pregunta, decide si usar tools, qué tools usar y genera la respuesta final. Esto mezcla responsabilidades críticas. En software tradicional esto sería equivalente a construir un modulo monolítico que hace todo. En Ingeniería de Software, sabemos que eso genera problema de mantenibilidad, evolutividad y monitoreabilidad.

- Aunque tengamos recursion_limit, el problema estructural sigue que el modelo puede volver a pedir tools y puede no converger a una respuesta final clara. 

- El agente actual es “generalista”. Un solo nodo hace todo el razonamiento. No hay separación de roles ni etapas claras.


**Solución**

En lugar de un agente que “decide todo”, construimos un workflow explícito de decisiones y etapas. Un agentic workflow es un sistema donde el LLM y las tools se organizan como un flujo estructurado de pasos especializados.

Pasamos de:
1. Agente simple: User → LLM → (maybe tool) → LLM → Answer

2. Agentic Workflow: User  → Planner (define pasos) → Executor (usa tools si es necesario) → Writer (redacta respeusta final) → Output

In [52]:
from typing import TypedDict, Annotated
import requests

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

In [53]:
class State(TypedDict):
    messages: Annotated[list, add_messages]
    plan: str

In [54]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [55]:
tools

[StructuredTool(name='calculadora', description='Solo evalúa expresiones matemáticas simples.', args_schema=<class 'langchain_core.utils.pydantic.calculadora'>, func=<function calculadora at 0x7a89777f3c40>),
 StructuredTool(name='usd_a_clp', description='Obtiene el valor actual del dólar (USD) en pesos chilenos (CLP).', args_schema=<class 'langchain_core.utils.pydantic.usd_a_clp'>, func=<function usd_a_clp at 0x7a897759fb00>)]

In [56]:
def planner_node(state: State):
    print("=== PLANNER ===")

    prompt = f"""
Eres un planner.

Divide el problema en pasos. Solo devuelve una lista de paso a seguir de manera muy sintetica, con numerotación.

Pregunta del usuario:
{state['messages'][-1].content}
"""
    plan = llm.invoke(prompt).content
    print("PLAN:\n"+plan)
    return {"plan": plan} ### aqui se actualiza el State

In [57]:
tools = [usd_a_clp, calculadora]

llm_with_tools = llm.bind_tools(tools)

def agent_node(state):
    print("=== EXECUTOR ===")

    response = llm_with_tools.invoke(state["messages"])

    print("-TOOL CALLS:", response.tool_calls)

    return {"messages": [response]}

In [58]:
from langgraph.prebuilt import ToolNode

tool_node = ToolNode(tools)

In [59]:
def should_continue(state: State):
    last = state["messages"][-1]

    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"

    return "writer"

In [60]:
def writer_node(state: State):
    print("=== WRITER ===")

    prompt = f"""
Eres un redactor pedagógico. Solo dame las respuestas de manera organizada y síntetica.

Plan inicial:
{state['plan']}

Memoria etapas anteriores:
{state['messages']}

"""
    
    final = llm.invoke(prompt)


    return {"messages": [final]}

In [61]:
from langgraph.graph import StateGraph, END


graph = StateGraph(State)

graph.add_node("planner", planner_node)
graph.add_node("agent", agent_node)
graph.add_node("tools", tool_node)
graph.add_node("writer", writer_node)

In [62]:
graph.set_entry_point("planner")

graph.add_edge("planner", "agent")

graph.add_conditional_edges(
    "agent",
    should_continue,
    {
        "tools": "tools",
        "writer": "writer"
    }
)

graph.add_edge("tools", "agent")

graph.add_edge("writer", END)



In [63]:
app = graph.compile()

In [64]:
result = app.invoke({
    "messages": [
        {"role": "user", "content": "¿Cuánto es 25 * 37 y cuánto es en dólares 10000 CLP? No uses simbolos Latex. Cual es la capital de Francia"}
    ]
})

print(result["messages"][-1].content)

=== PLANNER ===
PLAN:
1. Calcular 25 * 37.
2. Convertir 10000 CLP a dólares.
3. Investigar la capital de Francia.
=== EXECUTOR ===
-TOOL CALLS: [{'name': 'calculadora', 'args': {'expresion': '25 * 37'}, 'id': 'call_O8xzVk5RIp0WVKNTFhYDlcPT', 'type': 'tool_call'}, {'name': 'usd_a_clp', 'args': {}, 'id': 'call_3SGWvWjFln4K50spJOMWcEUO', 'type': 'tool_call'}]
=== EXECUTOR ===
-TOOL CALLS: []
=== WRITER ===
1. **Cálculo de 25 * 37**: 925.
2. **Conversión de 10,000 CLP a dólares**: Aproximadamente 11.24 USD (1 USD = 890.63 CLP).
3. **Capital de Francia**: París.


# Conclusión de esta sesión: de LLM a workflows agentic

En esta sesión pasamos de usar un modelo de lenguaje como un sistema aislado de respuesta, a construir un workflow estructurado con LangGraph compuesto por planner, executor con tools y writer. Esto representa un cambio importante: el modelo deja de ser una función única y pasa a formar parte de un sistema con flujo de control.

Sin embargo, este enfoque tiene limitaciones importantes.

Primero, el planner genera planes en lenguaje natural, lo que significa que no son estructuras ejecutables ni verificables. Son útiles como guía, pero no pueden garantizar consistencia ni control estricto sobre la ejecución.

Segundo, el executor no está obligado a seguir el plan ni a usar herramientas. Aunque las tools estén disponibles, el modelo puede resolver partes del problema sin invocarlas o incluso simular resultados. Esto reduce la confiabilidad del sistema.

Tercero, no existe verificación explícita del proceso. El sistema no valida si el plan se cumplió, si las herramientas se usaron correctamente o si los resultados son consistentes entre pasos. Esto hace que el flujo siga siendo parcialmente opaco.

Finalmente, todos los nodos siguen produciendo lenguaje natural, lo que significa que la lógica del sistema está distribuida en texto y no en estructuras formales ejecutables.

Estos límites son normales en esta etapa. Corresponden a lo que se puede llamar un workflow agentic inicial.

La evolución natural de este enfoque es avanzar hacia sistemas más estructurados. En el siguiente nivel, los planes dejan de ser texto libre y pasan a ser estructuras explícitas (listas o JSON). Los agentes dejan de ser genéricos y pasan a tener roles más definidos. El executor se convierte en un sistema paso a paso que ejecuta acciones verificables.

A partir de ahí, se introducen sistemas multi-agente, donde distintos agentes se especializan en roles como planificación, investigación, ejecución y redacción. En particular, el researcher agent aparece como un componente encargado de recopilar información y evidencia antes de que otros agentes tomen decisiones.

Finalmente, en sistemas de producción se agregan capas adicionales de control: observabilidad, trazabilidad, límites de costo, manejo de errores, seguridad en herramientas y evaluación sistemática del comportamiento del sistema.

En resumen, lo construido en esta sesión es el primer paso hacia sistemas agentic modernos: pasar de un modelo que responde a un sistema que coordina decisiones, herramientas y flujo de ejecución.

# Próxima sesión: Integración de Agentic Workflows con RAG

En la próxima clase veremos cómo extender los workflows agentic construidos en LangGraph (planner → agent → tools → writer) para incorporar recuperación de información externa mediante RAG.

La idea es pasar de agentes que solo razonan con herramientas, a agentes que también pueden consultar conocimiento externo de forma dinámica y controlada.